# 04 - Liwa Oasis Sentinel-2 Time Series & Phenology Analysis

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook processes multi-temporal Sentinel-2 L2A surface reflectance imagery over Liwa Oasis (`[53.5, 22.8, 54.2, 23.3]`) to construct vegetation index time series (NDVI, SAVI, SI) for crop phenology and seasonal stress detection.

The mechanistic justification for this step is: Multi-temporal index stacking tracks date palm canopy phenology and separates seasonal physiological variation from stress induced by groundwater salinization.

In [ ]:
# Step 1: Environment Setup & Data Discovery
# The mechanistic justification for this step is: Dynamically discovering mounted Kaggle dataset paths ensures multi-temporal Sentinel-2 scenes load seamlessly across Kaggle environments.
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

def find_files(pattern):
    return sorted(glob.glob(pattern, recursive=True))

s2_b04_files = find_files("/kaggle/input/**/B04.tif") or find_files("../data/**/B04.tif")
s2_b08_files = find_files("/kaggle/input/**/B08.tif") or find_files("../data/**/B08.tif")

print("=== Sentinel-2 Time Series Discovery ===")
print(f"B04 (Red) Band Files Found: {len(s2_b04_files)}")
print(f"B08 (NIR) Band Files Found: {len(s2_b08_files)}")


In [ ]:
# Step 2: Extract Time Series Vegetation Index Metrics
# The mechanistic justification for this step is: Computing mean and maximum NDVI/SAVI values across sequential dates tracks canopy health dynamics.
records = []
for i, (b04_path, b08_path) in enumerate(zip(s2_b04_files, s2_b08_files)):
    scene_name = os.path.basename(os.path.dirname(b04_path))
    with rasterio.open(b04_path) as r_src, rasterio.open(b08_path) as nir_src:
        red = r_src.read(1).astype(np.float32) / 10000.0
        nir = nir_src.read(1).astype(np.float32) / 10000.0
        min_h, min_w = min(red.shape[0], nir.shape[0]), min(red.shape[1], nir.shape[1])
        red, nir = red[:min_h, :min_w], nir[:min_h, :min_w]
        
        ndvi = (nir - red) / (nir + red + 1e-8)
        savi = ((nir - red) / (nir + red + 0.5)) * 1.5
        
        records.append({
            'scene': scene_name,
            'ndvi_mean': float(np.nanmean(ndvi)),
            'ndvi_max': float(np.nanmax(ndvi)),
            'savi_mean': float(np.nanmean(savi)),
            'savi_max': float(np.nanmax(savi))
        })

df_ts = pd.DataFrame(records)
print("=== Multi-Temporal Vegetation Summary ===")
print(df_ts.to_string(index=False))

if not df_ts.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df_ts['scene'], df_ts['ndvi_max'], marker='o', color='green', linewidth=2, label='Max NDVI (Canopy Peak)')
    ax.plot(df_ts['scene'], df_ts['savi_max'], marker='s', color='goldenrod', linewidth=2, label='Max SAVI (Soil Adjusted)')
    ax.set_title("Sentinel-2 Canopy Health Phenology Curve (Liwa Oasis)")
    ax.set_ylabel("Index Value")
    plt.xticks(rotation=30, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
